# 06 — Base SLM evaluation (Colab GPU, MANDATORY before fine-tuning)

## Prerequisites
1. Run `01_environment_check` first to verify GPU
2. Mount Google Drive for persistence
3. Copy repo to Drive: `/content/drive/MyDrive/MaintainAI/code`

## Process
1. Load base model (Qwen2.5-3B-Instruct) via HFBackend (4-bit)
2. Score `data/slm/test.jsonl` (100 held-out scenarios) via `src.slm_eval.run_harness`
3. Save metrics to `reports/metrics_base.json`
4. Paste numbers into `docs/EVALUATION.md`

**Do NOT fine-tune before recording this baseline.**

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install dependencies
!pip install -q transformers accelerate bitsandbytes peft 2>&1 | tail -1

# Add repo to path
import sys
sys.path.insert(0, '/content/drive/MyDrive/MaintainAI/code')

# Verify GPU
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), 'GB')

In [ ]:
# Load base model and run evaluation
import json
from src.slm_service import SLMService, HFBackend
from src.slm_eval import run_harness

# Configure base model (4-bit quantized)
svc = SLMService(
    config={'slm': {'model_name': 'Qwen/Qwen2.5-3B-Instruct'}},
    backend=HFBackend('Qwen/Qwen2.5-3B-Instruct', quantization='4bit')
)

# Load test scenarios (100 decision-point examples from NASA test fleet)
test = [json.loads(l) for l in open('data/slm/test.jsonl')]
print(f'Loaded {len(test)} test scenarios')

def analyze_fn(e):
    raw = svc.backend.generate(e['system'] + '\n' + e['user'])
    from src.slm_service import extract_json
    from src.schemas import SLMAnalysis
    obj = extract_json(raw)
    try:
        SLMAnalysis.model_validate(obj or {})
        return {**obj, 'meta': {'backend': svc.version, 'valid': True}}
    except Exception as ex:
        return {'meta': {'backend': svc.version, 'valid': False, 'reason': str(ex)[:200]}}

# Run harness
m = run_harness(test, analyze_fn)
print(json.dumps(m, indent=1))

# Save metrics
import os
os.makedirs('/content/drive/MyDrive/MaintainAI/reports', exist_ok=True)
with open('/content/drive/MyDrive/MaintainAI/reports/metrics_base.json', 'w') as f:
    json.dump({**m, 'backend': svc.version}, f, indent=1)
print('\n✓ Saved: /content/drive/MyDrive/MaintainAI/reports/metrics_base.json')

## After running, copy metrics to local repo:
```bash
cp /content/drive/MyDrive/MaintainAI/reports/metrics_base.json reports/
# Then run: python scripts/compare_slm.py
```

Then update `docs/EVALUATION.md` with the measured base model numbers.